# 02 — English-only GA, GD, NPO unlearning on MQuAKE-T + MQuAKE-CF original facts

Requires **saved Kaggle output from Notebook 01** containing `mquake_t_records.json` and `full_sft_adapter`. Forget examples are the original **English** direct questions for the selected first-hop facts. Supporting facts (including for forget cases) remain in the retain objective. German target questions and held-out target multi-hop questions **never** enter the unlearning loss.

NPO uses sequence log-probability relative to an initial frozen SFT reference, cached before updates; checkpoints are trained independently from the **same** SFT adapter. This is exploratory adapter-level unlearning, not certified data deletion.

In [ ]:
%pip install -q "transformers>=4.48,<5" "peft>=0.15,<0.19" "accelerate>=1.2,<2" "bitsandbytes>=0.45" "sentencepiece>=0.2" "safetensors>=0.5" "pandas>=2" "requests>=2.31" "wandb>=0.31,<1"

In [ ]:
import os,gc,json,random,re,time,pathlib,unicodedata,math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel

MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
SEED=20261026
MAX_SEQ=384
ROOT=pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle').exists() else './kaggle_output')
ROOT.mkdir(parents=True,exist_ok=True)
SYSTEM='Answer the factual question in English with only the short entity name or phrase. Do not explain. If unsure, say UNKNOWN.'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def require_gpu():
    assert torch.cuda.is_available(),'Enable a GPU accelerator in Kaggle settings.'
    print('GPU devices:',[(i,torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])

def cleanup():
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def get_tokenizer():
    t=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
    if t.pad_token is None:t.pad_token=t.eos_token
    return t

def get_backbone():
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=q,torch_dtype=torch.float16,device_map={'':0},attn_implementation='sdpa')

def make_qa(e,tok):
    prompt=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':e['q']}],tokenize=True,add_generation_prompt=True)
    if isinstance(prompt,torch.Tensor):prompt=prompt.tolist()
    ans=tok.encode(e['a'],add_special_tokens=False)+[tok.eos_token_id]
    if len(prompt)+len(ans)>MAX_SEQ:
        raise ValueError(f'Example exceeds MAX_SEQ={MAX_SEQ}, increase limit or filter: {e["q"][:60]}')
    ids=prompt+ans
    return {'input_ids':torch.tensor([ids],dtype=torch.long),
            'labels':torch.tensor([[-100]*len(prompt)+ans],dtype=torch.long),
            'attention_mask':torch.ones((1,len(ids)),dtype=torch.long)}

def to_gpu(b):return {k:v.to('cuda:0') for k,v in b.items()}

def seq_logp(model,b):
    outs=model(input_ids=b['input_ids'],attention_mask=b['attention_mask'])
    logits=outs.logits[:,:-1,:].float()
    labels=b['labels'][:,1:]
    mask=labels.ne(-100)
    selected=torch.log_softmax(logits,dim=-1).gather(-1,labels.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    return (selected*mask).sum(dim=-1)

def generate_answer(model,tok,q,max_new_tokens=30):
    model.eval()
    prefix=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':q}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda:0')
    with torch.inference_mode():
        out=model.generate(input_ids=prefix,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id,eos_token_id=tok.eos_token_id)
    return tok.decode(out[0,prefix.shape[-1]:],skip_special_tokens=True).strip()

def normalize_answer(text):
    t=unicodedata.normalize('NFKC',str(text or '')).casefold().strip()
    t=re.sub(r'[^\w\s]', ' ',t)
    return re.sub(r'\s+',' ',t).strip()

def match_answer(response,gold,aliases=()):
    """Conservative exact/short-contained matching. Review outputs manually for publication."""
    p=normalize_answer(response)
    if not p or 'unknown' in p or 'i don t know' in p:return 0
    choices=[normalize_answer(x) for x in [gold]+list(aliases or []) if str(x).strip()]
    choices=[x for x in choices if x and len(x)>1]
    if p in choices:return 1
    if len(p)>75:return 0
    return int(any(re.search(r'(?<!\w)'+re.escape(c)+r'(?!\w)',p) is not None for c in choices if len(c)>2))

In [ ]:
require_gpu()
INPUT=pathlib.Path('/kaggle/input')
DATA_OVERRIDE=None
SFT_OVERRIDE=None

def unique_file(fname):
    fs=sorted(INPUT.rglob(fname))
    if len(fs)!=1:raise RuntimeError(f'Need exactly one {fname}; found: {fs}. Attach Notebook 01 saved output or set override.')
    return fs[0]
def unique_adapter(name):
    fs=sorted(p.parent for p in INPUT.rglob('adapter_config.json') if p.parent.name==name+'_adapter')
    if len(fs)!=1:raise RuntimeError(f'Need exactly one {name}_adapter, found: {fs}')
    return fs[0]
DATA=pathlib.Path(DATA_OVERRIDE) if DATA_OVERRIDE else unique_file('mquake_t_records.json')
SFT=pathlib.Path(SFT_OVERRIDE) if SFT_OVERRIDE else unique_adapter('full_sft')
pack=json.loads(DATA.read_text())
assert pack['dataset']=='MQuAKE-original-mixed' and pack['model_id']==MODEL_ID and pack['schema_version']==4
forget=pack['forget_train']
retain=pack['retain_reference']
assert forget and retain
assert tuple(pack["train_languages"])==("en","de")
assert tuple(pack["evaluation_languages"])==("en","de","ru","zh","tr","vi")
assert all(e['lang']=='en' and e['relation']=='target' for e in forget)
assert not any(e['split']=='forget' and e['relation']=='target' for e in retain)
print('Source SHA:',pack['source_sha256'],'Forget:',len(forget),'Retain:',len(retain))
# Prevent wasting the unlearning budget if the original SFT checkpoint cannot perform the tasks.
STRICT_SFT_SANITY=True
SANITY=unique_file('sft_sanity.csv')
sanity=pd.read_csv(SANITY)
sc=sanity[sanity['split']=='forget'].groupby(['language','probe'])['correct'].mean()
limits={('en','direct'):0.50,('de','direct'):0.50,('en','multi'):0.25,('de','multi'):0.25}
fail={key:(round(float(sc.get(key,float('nan'))),3),threshold) for key,threshold in limits.items()
      if pd.isna(sc.get(key,float('nan'))) or float(sc.get(key,float('nan')))<threshold}
if fail:
    print('Full SFT did not meet the exploratory prerequisite. Observed and required:',fail)
    if STRICT_SFT_SANITY:
        raise RuntimeError('STOP: improve source-fact learning and original two-hop competence in Notebook 01, then rerun. Set STRICT_SFT_SANITY=False only for debugging.')
tokenizer=get_tokenizer()
assert max(make_qa(e,tokenizer)['input_ids'].shape[-1] for e in forget+retain)<=MAX_SEQ

In [ ]:
# W&B / CoreWeave Forge integration.
# Kaggle's Secrets HTTP service can fail independently of user credentials.
# Preserve all logs offline if the secret cannot be read during a Kaggle API run.
import os
from pathlib import Path
import wandb

WANDB_ENTITY = "crosslingual"
WANDB_PROJECT = "multilingual_deepunlearning"
WANDB_DASHBOARD = "https://forge.coreweave.com/wandb/crosslingual/multilingual_deepunlearning"
WANDB_GROUP = "mquake-t-20261026"
WANDB_BASE_URL = "https://api.forge.coreweave.com"
os.environ.setdefault("WANDB_BASE_URL", WANDB_BASE_URL)
WANDB_OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle").exists() else "./wandb_output")
WANDB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def start_tracking(stage, name, config=None):
    """Track online when Kaggle key works, otherwise keep syncable offline runs.

    Does not print the key. Offline runs are saved in /kaggle/working/wandb
    and can be synced later after authenticating to CoreWeave Forge.
    """
    token = os.environ.get("WANDB_API_KEY", "").strip()
    secret_problem = None
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = (UserSecretsClient().get_secret("WANDB_API_KEY") or "").strip()
        except Exception as exc:
            secret_problem = type(exc).__name__
    if token:
        os.environ["WANDB_API_KEY"] = token
    mode = "online" if token else "offline"
    if mode == "offline":
        print(
            "[W&B] Kaggle WANDB_API_KEY not accessible"
            + (f" ({secret_problem})" if secret_problem else "")
            + ". Logging OFFLINE to /kaggle/working/wandb; training will continue."
        )
        print("[W&B] Enable the notebook's Kaggle Secrets permission for live graphs.")
    options = dict(
        entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
        job_type=stage, name=name, config=config or {}, mode=mode,
        dir=str(WANDB_OUTPUT_DIR),
        settings=wandb.Settings(init_timeout=60),
    )
    try:
        run = wandb.init(**options)
    except Exception as exc:
        if mode != "online":
            raise
        print(
            "[W&B] Online connection failed"
            f" ({type(exc).__name__}); using syncable offline logging instead."
        )
        try:
            wandb.finish(exit_code=1)
        except Exception:
            pass
        options["mode"] = "offline"
        run = wandb.init(**options)
    if run is None:
        raise RuntimeError("W&B returned no run in either online or offline mode.")
    print("[W&B]", "online" if options["mode"] == "online" else "offline",
          "run:", run.id, "| project:", WANDB_DASHBOARD)
    return run


In [ ]:
METHODS=['ga','gd','npo']
# Larger factual forget set requires more optimization steps than the 24-case pilot.
# 512 steps with ACCUM=2 samples about 1,024 forget instances per method.
# Random-with-replacement sampling means not every target is guaranteed exposure.
N_STEPS={'ga':512,'gd':512,'npo':512}
LR={'ga':1e-5,'gd':1e-5,'npo':2e-5}
RETAIN_WEIGHT=1.0
BETA=0.1
ACCUM=2

def new_sft():
    base=get_backbone()
    base.config.use_cache=False
    base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True)
    return PeftModel.from_pretrained(base,str(SFT),is_trainable=True)

def sample_indices(method):
    rng=random.Random(SEED+{'ga':11,'gd':23,'npo':37}[method])
    return [(rng.randrange(len(forget)),rng.randrange(len(retain))) for _ in range(N_STEPS[method]*ACCUM)]

In [ ]:
logs=[]
for method in METHODS:
    print('Starting',method)
    run = start_tracking("unlearning", method+"-english-forgetting", {"method": method, "seed": SEED, "dataset": "MQuAKE-original-mixed", "source_sha256": pack["source_sha256"], "steps": N_STEPS[method], "lr": LR[method], "beta": BETA, "retain_weight": RETAIN_WEIGHT, "forget_examples": len(forget), "retain_examples": len(retain)})
    run.define_metric("train/optimizer_step")
    run.define_metric("train/*", step_metric="train/optimizer_step")
    model=new_sft()
    model.eval()
    fixed={}
    if method=='npo':
        with torch.inference_mode():
            for i,e in enumerate(forget):
                fixed[i]=seq_logp(model,to_gpu(make_qa(e,tokenizer))).detach().float().cpu()
        assert len(fixed)==len(forget)
    model.train()
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR[method])
    optimizer.zero_grad(set_to_none=True)
    for j,(fi,ri) in enumerate(sample_indices(method)):
        fbatch=to_gpu(make_qa(forget[fi],tokenizer))
        if method=='npo':
            cur=seq_logp(model,fbatch)
            ref=fixed[fi].to(cur.device)
            forget_loss=-(2/BETA)*F.logsigmoid(-BETA*(cur-ref)).mean()
        else:
            forget_loss=-model(**fbatch).loss
        if method!='ga':
            rbatch=to_gpu(make_qa(retain[ri],tokenizer))
            retain_loss=model(**rbatch).loss
            loss=forget_loss+RETAIN_WEIGHT*retain_loss
        else:
            retain_loss=None
            loss=forget_loss
        (loss/ACCUM).backward()
        if (j+1)%ACCUM==0:
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step();optimizer.zero_grad(set_to_none=True)
            step=(j+1)//ACCUM
            step_metrics={"train/optimizer_step": step, "train/total_loss": float(loss.detach()), "train/forget_loss": float(forget_loss.detach()), "train/learning_rate": LR[method], "system/gpu_peak_gib": torch.cuda.max_memory_allocated(0)/1024**3}
            if retain_loss is not None: step_metrics["train/retain_loss"] = float(retain_loss.detach())
            run.log(step_metrics)
            logs.append({'method':method,'step':step,'loss':float(loss.detach()),
                         'forget_loss':float(forget_loss.detach()),
                         'retain_loss':float(retain_loss.detach()) if retain_loss is not None else None})
            if step==1 or step%8==0:print(method,step,'loss',round(float(loss.detach()),4))
    model.save_pretrained(ROOT/f'{method}_adapter',safe_serialization=True)
    tokenizer.save_pretrained(ROOT/f'{method}_adapter')
    print('Saved',ROOT/f'{method}_adapter')
    # Tiny, fixed pre-registered validation probes only, not test-language tuning.
    # The final multilingual measurements belong exclusively to Notebook 03.
    model.eval()
    check=[]
    for rec in [r for r in pack["records"] if r["split"]=="forget"][:24]:
        for lang in ("en","de"):
            answer=generate_answer(model,tokenizer,rec["target_question_"+lang])
            check.append({
                "method":method,"source_fact":rec["fact_id"],"language":lang,
                "direct_correct":match_answer(answer,rec["target_answer"],rec["target_aliases"]),
                "prediction":answer,
            })
    mini=pd.DataFrame(check)
    mini.to_csv(ROOT/f"{method}_direct_debug.csv",index=False)
    scores=mini.groupby("language")["direct_correct"].mean().to_dict()
    for language,accuracy in scores.items():
        run.log({"debug/"+language+"_direct_accuracy_after_unlearning":float(accuracy)})
    print("Debug 24-target direct accuracy after unlearning:",scores)
    run.summary["train/final_loss"] = float(loss.detach())
    run.summary["train/optimizer_steps"] = N_STEPS[method]
    run.finish()
    del optimizer,model,loss,forget_loss,retain_loss,fbatch
    if method!='ga':del rbatch
    if method=='npo':del cur,ref
    cleanup()
pd.DataFrame(logs).to_csv(ROOT/'unlearning_training_log.csv',index=False)
(ROOT/'unlearning_manifest.json').write_text(json.dumps({
    'dataset':'MQuAKE-original-mixed','source_sha256':pack['source_sha256'],'source_sft':str(SFT),
    'methods':METHODS,'steps':N_STEPS,'lr':LR,'beta':BETA,'retain_weight':RETAIN_WEIGHT,'seed':SEED,
    'unlearning_language':'en','unlearning_probe':'first_hop_direct',
    'evaluation_two_hops_seen_during_sft_for_forget_cases':False
},indent=2))
print('Notebook 02 complete. Save Kaggle version with adapters.')

**Next:** attach **both** saved Notebook 01 and Notebook 02 outputs to Notebook 03. These are first-pass hyperparameters, not tuned literature reproductions; select strength on a held-out development split using English forgetting and retain utility before final claims.